# Teste Final do Pipeline Multimodal com LangGraph + AWS S3

Este notebook realiza a validação completa do pipeline multimodal desenvolvido para apoio à triagem clínica preventiva em saúde feminina.

O fluxo integra:
- upload de arquivos multimodais no Amazon S3;
- download automático pelo LangGraph;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa com LLM.

O pipeline utiliza:
- AWS S3;
- AWS Rekognition;
- YOLOv8;
- OpenCV;
- Librosa;
- LangGraph;
- RAG;
- Mistral via Ollama.

Os testes utilizam arquivos da base pública RAVDESS, composta por atores e emoções simuladas, evitando uso de dados reais de pacientes.

A análise possui finalidade exclusivamente educacional e de apoio à triagem preventiva.

In [1]:
import sys
print(sys.executable)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Scripts\python.exe


In [2]:
!{sys.executable} -m pip install moviepy imageio imageio-ffmpeg


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!{sys.executable} -m pip show moviepy

Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Lib\site-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [4]:
from moviepy import VideoFileClip

print("MoviePy funcionando.")

MoviePy funcionando.


In [5]:
import sys
sys.path.append("..")

import os
import boto3

from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from src.workflows.langgraph_flow import (
    build_medical_assistant_graph
)

from src.llm.ollama_client import (
    get_llm
)

from src.rag.vector_store import (
    load_vector_store
)

from src.rag.retriever import (
    get_retriever,
    retrieve_context
)

from src.multimodal.media_utils import (
    extract_audio_from_video
)

In [6]:
llm = get_llm(
    model_name="mistral",
    temperature=0.2
)

vector_store = load_vector_store(
    "../data/vector_store"
)

retriever = get_retriever(
    vector_store,
    k=3
)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

print("LangGraph compilado com sucesso.")

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store
LangGraph compilado com sucesso.


### CONFIGURAÇÃO AWS
Nesta etapa são carregadas as variáveis de ambiente e a configuração do Amazon S3 utilizada para armazenamento temporário dos arquivos multimodais.

In [7]:
load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

print("Bucket:", bucket)
print("Region:", region)

Bucket: postechfase4
Region: us-east-1


In [8]:
s3 = boto3.client(
    "s3",
    region_name=region
)

print("Cliente S3 criado com sucesso.")

Cliente S3 criado com sucesso.



### ESCOLHA DO CENÁRIO

Nesta etapa é selecionado o cenário multimodal utilizado para teste.

Os vídeos utilizados são provenientes de bancos públicos de mídia e representam situações simuladas de interação emocional. Os áudios utilizados foram criados de forma sintética em português para validar diferentes níveis de risco emocional sem uso de dados reais de pacientes.

Cenários sugeridos:

- baixo_risco
  - expressão neutra ou positiva
  - fala tranquila
  - ausência de palavras de alerta

- risco_medio
  - sinais leves de tristeza, cansaço ou preocupação
  - hesitação na fala
  - desconforto emocional moderado

- risco_alto
  - sinais aparentes de medo ou sofrimento
  - fala com palavras de alerta
  - possível tensão emocional

A combinação entre vídeo e áudio permite validar:
- análise visual;
- análise acústica;
- interpretação textual;
- fusão multimodal;
- geração automática de alertas preventivos.

## Cenário 1 -> Happy

In [88]:
videoHappy = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\HappyPregnantWoman.mp4"

print("Vídeo existe?", os.path.exists(videoHappy))
print(videoHappy)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\HappyPregnantWoman.mp4


## EXTRAÇÃO DO ÁUDIO

O pipeline possui suporte para extração automática de áudio a partir de arquivos de vídeo. Essa funcionalidade permite processar vídeos contendo fala original, separando automaticamente a trilha sonora para posterior análise acústica e textual.

A extração é realizada utilizando bibliotecas de processamento multimídia integradas ao projeto.

Nesta demonstração, entretanto, a etapa de extração automática não será utilizada, pois:
- os vídeos selecionados para análise visual não possuem áudio original;
- áudios sintéticos em português foram criados separadamente para simular diferentes cenários emocionais;
- essa abordagem permite maior controle sobre os testes multimodais e evita uso de dados reais de pacientes.

Mesmo não sendo utilizada neste cenário específico, a funcionalidade permanece implementada e disponível para futuros testes com vídeos completos contendo áudio integrado.

In [90]:
audio_output_extract = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_happy_audio_female.wav"
)

extract_audio_from_video(
    video_path=str(video),
    output_audio_path=str(audio_output_extract)
)

print("Áudio extraído com sucesso.")

Áudio extraído com sucesso.


In [92]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\happy.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\happy.wav


### UPLOAD S3

Nesta etapa os arquivos multimodais são enviados para o Amazon S3.

O pipeline LangGraph realizará o download automático durante a execução.

In [93]:
video_s3_key = (
    "inputs/videos/HappyPregnantWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/happy.wav"
)

s3.upload_file(
    str(video),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [94]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [95]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/HappyPregnantWoman.mp4 - 38292439 bytes
inputs/audios/happy.wav - 1097694 bytes


### BUILD LANGGRAPH

Nesta etapa o LangGraph é compilado para execução do pipeline multimodal integrado.

In [96]:
app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever
)

print("LangGraph compilado com sucesso.")

LangGraph compilado com sucesso.


### CRIAÇÃO DO ESTADO

O estado contém:
- pergunta do usuário;
- chaves dos arquivos multimodais no S3;
- idioma do áudio;
- identificação do cenário.

In [97]:
state = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "happy_woman1"
}

### EXECUÇÃO DO PIPELINE

O pipeline executa automaticamente:
- download dos arquivos do S3;
- análise de vídeo;
- análise de áudio;
- fusão multimodal;
- geração de resposta;
- alerta preventivo.

In [98]:
result = app.invoke(state)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


### RESULTADO DO VÍDEO

Nesta etapa é exibido o resultado da análise visual.

In [99]:
pprint(result["video_result"])

{'flags': ['person_detected', 'face_detected', 'multiple_people_detected'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'O vídeo apresentou múltiplas pessoas no mesmo frame, '
                    'indicando maior complexidade contextual da cena.',
                    'A análise facial não identificou sinais visuais '
                    'relevantes de desconforto emocional.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '


In [100]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- O vídeo apresentou múltiplas pessoas no mesmo frame, indicando maior complexidade contextual da cena.
- A análise facial não identificou sinais visuais relevantes de desconforto emocional.


### RESULTADO DO ÁUDIO

Nesta etapa é exibido o resultado da análise acústica e textual do áudio.

In [101]:
pprint(result["audio_result"])

{'detected_categories': {},
 'flags': ['voice_instability', 'elevated_voice_tension', 'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. Não '
                   'foram identificadas palavras-chave clínicas de alerta na '
                   'transcrição. Os sinais observados vieram principalmente de '
                   'características acústicas da voz, como variação de pitch, '
                   'pausas, energia ou intensidade vocal. Como a transcrição '
                   'não contém termos clínicos de alerta, esses sinais devem '
                   'ser tratados como evidências complementares de baixa '
                   'confiança. A intensidade vocal foi classificada como alta. '
                   'Foram registradas oscilações vocais, que podem indicar '
                   'variação emocional, mas isoladamente não confirmam '
                   'ansiedade ou tensão. Foram identificadas pausas ou '
                   'hesitações na 

In [102]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. Não foram identificadas palavras-chave clínicas de alerta na transcrição. Os sinais observados vieram principalmente de características acústicas da voz, como variação de pitch, pausas, energia ou intensidade vocal. Como a transcrição não contém termos clínicos de alerta, esses sinais devem ser tratados como evidências complementares de baixa confiança. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gra

### RESULTADO MULTIMODAL

Nesta etapa é exibido o resultado da fusão multimodal entre áudio e vídeo.

In [130]:
pprint(result["multimodal_result"])

{'alert': False,
 'audio_risk_level': 'alto',
 'audio_score': 0.82,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'multiple_people_detected',
                       'culpa',
                       'voice_instability',
                       'elevated_voice_tension',
                       'speech_hesitation'],
 'evidences': ['person_detected',
               'face_detected',
               'multiple_people_detected',
               'culpa',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.53,
 'fusion_strategy': 'audio_60_video_40',
 'interpretation': ['A análise de áudio apresentou sinais relevantes de '
                    'possível ansiedade, agitação ou fadiga emocional.',
                    'A análise de vídeo apresentou baixo nível de risco '
                    'visual.'],
 'limitations': ['A análise multimodal é apenas apoio à triagem cl

In [104]:
for item in result["multimodal_result"].get(
    "interpretation",
    []
):
    print("-", item)

- A análise de áudio apresentou baixo nível de risco. Os sinais acústicos detectados foram considerados fracos e complementares, sem evidência textual clínica de alerta.
- A análise de vídeo apresentou baixo nível de risco visual.


### RESPOSTA FINAL DO AGENTE

O modelo de linguagem gera uma resposta interpretativa utilizando:
- contexto multimodal;
- resultados da fusão;
- informações recuperadas via RAG.

A resposta possui finalidade exclusivamente educacional.

In [105]:
print(result["answer"])

 1. Resumo da avaliação: Não foram identificados sinais de risco emocional significativos na análise multimodal realizada.
2. Evidências observadas: A análise apresentou baixo nível de risco visual e acústico, sem evidência textual clínica de alerta. Os sinais detectados foram considerados fracos e complementares.
3. Nível de risco: Baixo
4. Recomendação: Não foram identificados sinais críticos. Recomenda-se manter observação e buscar orientação profissional apenas em caso de agravamento ou persistência dos sintomas.
5. Limitações da análise: A análise multimodal é apenas apoio à triagem clínica. O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico. Expressões faciais indicam apenas emoções aparentes. Sinais de áudio e vídeo devem ser interpretados como evidências complementares. A confirmação clínica depende de avaliação profissional.


### CONCLUSÃO

O teste validou:
- integração multimodal;
- upload e download via Amazon S3;
- processamento de vídeo;
- processamento de áudio;
- análise vocal;
- fusão multimodal;
- geração automática de respostas;
- integração completa do LangGraph.

As análises possuem finalidade educacional e de apoio à triagem preventiva, não representando diagnóstico médico ou psicológico.

### LIMPEZA S3

Após os testes, os arquivos podem ser removidos do S3 para evitar armazenamento desnecessário.

In [106]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/HappyPregnantWoman.mp4
Removido do S3: inputs/audios/happy.wav


## Cenário 2 -> SAD

In [107]:
videoSad = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\CryWoman.mp4"

print("Vídeo existe?", os.path.exists(videoHappy))
print(videoSad)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\CryWoman.mp4


In [108]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\sad.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\sad.wav


In [109]:
video_s3_key = (
    "inputs/videos/CryWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/sad.wav"
)

s3.upload_file(
    str(videoHappy),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [110]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [111]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/CryWoman.mp4 - 12282853 bytes
inputs/audios/sad.wav - 2268894 bytes


In [112]:
stateSad = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "sad_woman"
}

In [113]:
result = app.invoke(stateSad)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


In [114]:
pprint(result["video_result"])

{'flags': ['person_detected', 'face_detected'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'A análise facial não identificou sinais visuais '
                    'relevantes de desconforto emocional.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '
                 'substitui avaliação clínica especializada.',
                 'Em vídeos curtos ou bases sintéticas como RAVDESS, a emoção '
                 'pode variar rap

In [115]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- A análise facial não identificou sinais visuais relevantes de desconforto emocional.


In [116]:
pprint(result["audio_result"])

{'detected_categories': {'depressao_pos_parto_ou_sofrimento_emocional': ['não '
                                                                         'tenho '
                                                                         'vontade']},
 'flags': ['não tenho vontade',
           'voice_instability',
           'elevated_voice_tension',
           'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. A '
                   'análise textual identificou categorias associadas a '
                   'depressao_pos_parto_ou_sofrimento_emocional. Os principais '
                   'indicadores textuais e vocais encontrados foram: não tenho '
                   'vontade, voice_instability, elevated_voice_tension, '
                   'speech_hesitation. A intensidade vocal foi classificada '
                   'como alta. Foram registradas oscilações vocais, que podem '
                   'indicar variação emocional, mas isoladamente não co

In [117]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. A análise textual identificou categorias associadas a depressao_pos_parto_ou_sofrimento_emocional. Os principais indicadores textuais e vocais encontrados foram: não tenho vontade, voice_instability, elevated_voice_tension, speech_hesitation. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gravação, interpretação do ator ou hesitação. Os sinais identificados não representam diagnóstico médico ou psicológ

In [118]:
pprint(result["multimodal_result"])

{'alert': True,
 'audio_risk_level': 'alto',
 'audio_score': 0.77,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'não tenho vontade',
                       'voice_instability',
                       'elevated_voice_tension',
                       'speech_hesitation'],
 'evidences': ['person_detected',
               'face_detected',
               'não tenho vontade',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.77,
 'fusion_strategy': 'audio_only',
 'interpretation': ['A análise de áudio apresentou sinais relevantes de '
                    'possível ansiedade, agitação ou fadiga emocional.'],
 'limitations': ['A análise multimodal é apenas apoio à triagem clínica.',
                 'O sistema não realiza diagnóstico médico, psicológico ou '
                 'psiquiátrico.',
                 'Expressões faciais indicam apenas emoções apa

In [119]:
for item in result["multimodal_result"].get(
    "interpretation",
    []
):
    print("-", item)

- A análise de áudio apresentou sinais relevantes de possível ansiedade, agitação ou fadiga emocional.


In [120]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal identificou sinais relevantes de possível ansiedade, agitação ou fadiga emocional na pessoa.
2. Evidências observadas: As evidências incluem expressões faciais como não tenho vontade e sinais de áudio como voice_instability, elevated_voice_tension e speech_hesitation.
3. Nível de risco: O nível de risco é alto.
4. Recomendação: Recomenda-se priorizar avaliação por profissional de saúde, especialmente se os sinais forem persistentes, intensos ou associados a sofrimento.
5. Limitações da análise:
   - A análise multimodal é apenas apoio à triagem clínica.
   - O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico.
   - Expressões faciais indicam apenas emoções aparentes.
   - Sinais de áudio e vídeo devem ser interpretados como evidências complementares.
   - A confirmação clínica depende de avaliação profissional.


In [121]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/CryWoman.mp4
Removido do S3: inputs/audios/sad.wav


## Interpretação do Cenário


## Cenário 3 -> FEAR

In [122]:
videoFear = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\FearWoman.mp4"

print("Vídeo existe?", os.path.exists(videoFear))
print(videoFear)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\FearWoman.mp4


In [123]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\fear.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\fear.wav


In [124]:
video_s3_key = (
    "inputs/videos/FearWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/fear.wav"
)

s3.upload_file(
    str(video),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [125]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [126]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/FearWoman.mp4 - 38292439 bytes
inputs/audios/fear.wav - 2194014 bytes


In [127]:
stateFear = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "fear_woman"
}

In [128]:
result = app.invoke(stateFear)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


In [131]:
pprint(result["video_result"])

{'flags': ['person_detected', 'face_detected', 'multiple_people_detected'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'O vídeo apresentou múltiplas pessoas no mesmo frame, '
                    'indicando maior complexidade contextual da cena.',
                    'A análise facial não identificou sinais visuais '
                    'relevantes de desconforto emocional.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '


In [132]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- O vídeo apresentou múltiplas pessoas no mesmo frame, indicando maior complexidade contextual da cena.
- A análise facial não identificou sinais visuais relevantes de desconforto emocional.


In [133]:
pprint(result["audio_result"])

{'detected_categories': {'sinais_de_violencia_ou_medo': ['culpa']},
 'flags': ['culpa',
           'voice_instability',
           'elevated_voice_tension',
           'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. A '
                   'análise textual identificou categorias associadas a '
                   'sinais_de_violencia_ou_medo. Os principais indicadores '
                   'textuais e vocais encontrados foram: culpa, '
                   'voice_instability, elevated_voice_tension, '
                   'speech_hesitation. A intensidade vocal foi classificada '
                   'como alta. Foram registradas oscilações vocais, que podem '
                   'indicar variação emocional, mas isoladamente não confirmam '
                   'ansiedade ou tensão. Foram identificadas pausas ou '
                   'hesitações na fala, interpretadas apenas como sinal '
                   'acústico complementar. A análise acústica r

In [134]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. A análise textual identificou categorias associadas a sinais_de_violencia_ou_medo. Os principais indicadores textuais e vocais encontrados foram: culpa, voice_instability, elevated_voice_tension, speech_hesitation. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gravação, interpretação do ator ou hesitação. Os sinais identificados não representam diagnóstico médico ou psicológico. A análise de áudio é us

In [137]:
pprint(result["multimodal_result"])

{'alert': False,
 'audio_risk_level': 'alto',
 'audio_score': 0.82,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'multiple_people_detected',
                       'culpa',
                       'voice_instability',
                       'elevated_voice_tension',
                       'speech_hesitation'],
 'evidences': ['person_detected',
               'face_detected',
               'multiple_people_detected',
               'culpa',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.53,
 'fusion_strategy': 'audio_60_video_40',
 'interpretation': ['A análise de áudio apresentou sinais relevantes de '
                    'possível ansiedade, agitação ou fadiga emocional.',
                    'A análise de vídeo apresentou baixo nível de risco '
                    'visual.'],
 'limitations': ['A análise multimodal é apenas apoio à triagem cl

In [138]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/FearWoman.mp4
Removido do S3: inputs/audios/fear.wav
